In [1]:
# Python Standard Library
import sys
from os.path import join

# Community Modules
from tqdm import tqdm
import numpy as np
import pandas as pd

# My Modules
sys.path.insert(0, "../code")
import measure_signal as ms
import dataset_utils as du

rng = np.random.default_rng(1415)

2026-03-28 19:49:56.614915: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-28 19:49:56.635246: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-28 19:49:56.641071: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-28 19:49:56.656327: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-28 19:49:58.603109: W tensorflow/compiler/tf2

In [2]:
file_dataset = "../data/forSNR/deprecated_4/dataset.parquet"
file_signal_only = "../data/forSNR/deprecated_4/signal.parquet"
file_noise_only = "../data/forSNR/deprecated_4/noise.parquet"

df_dataset = pd.read_parquet(file_dataset)
df_signal = pd.read_parquet(file_signal_only)
df_noise = pd.read_parquet(file_noise_only)

df_dataset.loc[:, "noiseWindowBlu"] *= 2.5
df_dataset.loc[:, "noiseWindowRed"] *= 2.5

df_signal.loc[:, "noiseWindowBlu"] *= 2.5
df_signal.loc[:, "noiseWindowRed"] *= 2.5

df_noise.loc[:, "noiseWindowBlu"] *= 2.5
df_noise.loc[:, "noiseWindowRed"] *= 2.5

wvl, spectra, df_meta = du.unpack_dataset(df_dataset)
num_wvl = wvl.size
num_spectra = df_dataset.shape[0]

In [3]:
new_SNR_arr = np.empty(num_spectra)
new_S_arr = np.empty(num_spectra)
new_N_arr = np.empty(num_spectra)
new_signal_arr = np.empty((num_spectra, num_wvl))
new_noise_arr = np.empty((num_spectra, num_wvl))

In [4]:
for i in tqdm(range(num_spectra)):
    specsnr = ms.SpectrumSNR(
        df_dataset.loc[i, "SN Name"],
        df_dataset.loc[i, "SN Subtype"],
        df_dataset.loc[i, "Spectrum Phase"],
        wvl,
        df_dataset.loc[i, wvl.astype(str)].to_numpy())
    specsnr.execute_algorithm(df_dataset.loc[i])

    new_SNR_arr[i] = specsnr.SNR
    new_S_arr[i] = specsnr.S
    new_N_arr[i] = specsnr.N
    new_signal_arr[i] = specsnr.signal
    new_noise_arr[i] = specsnr.noise
        
    # df_dataset.loc[i, "SNR"] = specsnr.SNR
    # df_dataset.loc[i, "S (SNR)"] = specsnr.S
    # df_dataset.loc[i, "N (SNR)"] = specsnr.N

    # df_signal.loc[i, "SNR"] = specsnr.SNR
    # df_signal.loc[i, "S (SNR)"] = specsnr.S
    # df_signal.loc[i, "N (SNR)"] = specsnr.N
    # df_signal.loc[i, wvl.astype(str)] = specsnr.signal

    # df_noise.loc[i, "SNR"] = specsnr.SNR
    # df_noise.loc[i, "S (SNR)"] = specsnr.S
    # df_noise.loc[i, "N (SNR)"] = specsnr.N
    # df_noise.loc[i, wvl.astype(str)] = specsnr.noise

100%|██████████| 3574/3574 [19:44<00:00,  3.02it/s]


In [5]:
df_dataset["SNR"] = new_SNR_arr
df_dataset["S (SNR)"] = new_S_arr
df_dataset["N (SNR)"] = new_N_arr

df_signal["SNR"] = new_SNR_arr
df_signal["S (SNR)"] = new_S_arr
df_signal["N (SNR)"] = new_N_arr
df_signal[wvl.astype(str)] = new_signal_arr

df_noise["SNR"] = new_SNR_arr
df_noise["S (SNR)"] = new_S_arr
df_noise["N (SNR)"] = new_N_arr
df_noise[wvl.astype(str)] = new_noise_arr

In [6]:
df_dataset.to_parquet("../data/forSNR/dataset.parquet")
df_signal.to_parquet("../data/forSNR/signal.parquet")
df_noise.to_parquet("../data/forSNR/noise.parquet")